In [2]:
import os, gc
import numpy as np
import pandas as pd

DATA_DIR = "./data"
TARGET_COL = "amount_new_house_transactions"
CUT_YEAR, CUT_MONTH = 2024, 7

# ==== 2) OPTIMIZED DATA LOADING ====
def load_all_data(data_dir):
    """Load all datasets with optimized memory usage"""
    data = {}
    
    # Main datasets
    datasets = {
        'new': 'train/new_house_transactions.csv',
        'new_nb': 'train/new_house_transactions_nearby_sectors.csv',
        'pre': 'train/pre_owned_house_transactions.csv',
        'pre_nb': 'train/pre_owned_house_transactions_nearby_sectors.csv',
        'land': 'train/land_transactions.csv',
        'land_nb': 'train/land_transactions_nearby_sectors.csv',
        'city_idx': 'train/city_indexes.csv',
        'city_search': 'train/city_search_index.csv',
        'poi': 'train/sector_POI.csv',
        'test': 'test.csv'
    }
    
    for name, path in datasets.items():
        try:
            data[name] = pd.read_csv(os.path.join(data_dir, path))
            print(f"Loaded {name}: {data[name].shape}")
        except Exception as e:
            print(f"Error loading {name}: {e}")
    
    return data

# Load all data
all_data = load_all_data(DATA_DIR)

Loaded new: (5433, 11)
Loaded new_nb: (5360, 11)
Loaded pre: (5360, 6)
Loaded pre_nb: (5427, 6)
Loaded land: (5896, 6)
Loaded land_nb: (5025, 6)
Loaded city_idx: (7, 74)
Loaded city_search: (4020, 4)
Loaded poi: (86, 142)
Loaded test: (1152, 2)


In [3]:
# Merge multiple DataFrames sequentially on ['month', 'sector']
pros_data = all_data['new'].merge(all_data['new_nb'], on=['month', 'sector'], how='outer')\
    .merge(all_data['pre'], on=['month', 'sector'], how='outer', suffixes=('', '_pre'))\
    .merge(all_data['pre_nb'], on=['month', 'sector'], how='outer', suffixes=('', '_pre_nb'))\
    .merge(all_data['land'], on=['month', 'sector'], how='outer', suffixes=('', '_land'))\
    .merge(all_data['land_nb'], on=['month', 'sector'], how='outer', suffixes=('', '_land_nb'))
pros_data.merge(all_data['poi'],on=['sector'], how='outer')

,month,sector,num_new_house_transactions,area_new_house_transactions,price_new_house_transactions,amount_new_house_transactions,area_per_unit_new_house_transactions,total_price_per_unit_new_house_transactions,num_new_house_available_for_sale,area_new_house_available_for_sale,...,medical_health_rehabilitation_institution_dense,medical_health_first_aid_center_dense,medical_health_blood_donation_station_dense,medical_health_disease_prevention_institution_dense,medical_health_general_hospital_dense,medical_health_clinic_dense,education_training_school_education_middle_school_dense,education_training_school_education_primary_school_dense,education_training_school_education_kindergarten_dense,education_training_school_education_research_institution_dense
0,2019-Apr,sector 1,69.0,6935.0,38392.0,26626.68,101.0,385.89,141.0,12936.0,...,0.000113,0.000000,0.000000,8.600000e-07,0.000041,0.000038,0.000016,0.000028,0.000063,0.000014
1,2019-Aug,sector 1,54.0,6263.0,47573.0,29794.76,116.0,551.75,91.0,11076.0,...,0.000113,0.000000,0.000000,8.600000e-07,0.000041,0.000038,0.000016,0.000028,0.000063,0.000014
2,2019-Dec,sector 1,48.0,4585.0,50012.0,22931.65,96.0,477.74,293.0,36501.0,...,0.000113,0.000000,0.000000,8.600000e-07,0.000041,0.000038,0.000016,0.000028,0.000063,0.000014
3,2019-Feb,sector 1,24.0,2526.0,34846.0,8802.81,105.0,366.78,158.0,15814.0,...,0.000113,0.000000,0.000000,8.600000e-07,0.000041,0.000038,0.000016,0.000028,0.000063,0.000014
4,2019-Jan,sector 1,52.0,4906.0,28184.0,13827.14,94.0,265.91,159.0,15904.0,...,0.000113,0.000000,0.000000,8.600000e-07,0.000041,0.000038,0.000016,0.000028,0.000063,0.000014
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6427,2024-Jan,sector 96,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000048,0.000002,0.000008,3.850000e-06,0.000013,0.000079,0.000010,0.000010,0.000033,0.000013
6428,2024-Jul,sector 96,1.0,140.0,40079.0,561.19,140.0,561.19,1.0,195.0,...,0.000048,0.000002,0.000008,3.850000e-06,0.000013,0.000079,0.000010,0.000010,0.000033,0.000013
6429,2024-Jun,sector 96,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000048,0.000002,0.000008,3.850000e-06,0.000013,0.000079,0.000010,0.000010,0.000033,0.000013
6430,2024-Mar,sector 96,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000048,0.000002,0.000008,3.850000e-06,0.000013,0.000079,0.000010,0.000010,0.000033,0.000013


In [4]:
def extract_datetime_features(df, date_col='month'):
    """Extract comprehensive datetime features"""
    df = df.copy()
    
    if date_col in df.columns:
        # Extract year and month
        df[['Year', 'Month']] = df[date_col].str.split('-', expand=True)
        df['Year'] = df['Year'].astype(int)
        
        # Month mapping
        month_map = {'Jan':1, 'Feb':2, 'Mar':3, 'Apr':4, 'May':5, 'Jun':6,
                    'Jul':7, 'Aug':8, 'Sep':9, 'Oct':10, 'Nov':11, 'Dec':12}
        df['Month'] = df['Month'].map(month_map)
        
        # Create datetime index
        # df['date'] = pd.to_datetime(df['Year'].astype(str) + '-' + df['Month_num'].astype(str) + '-01')
        df['time_index'] = (df['Year'] - 2019) * 12 + df['Month']
        
        # Temporal features
        df['quarter'] = (df['Month'] - 1) // 3 + 1
        df['is_quarter_end'] = df['Month'].isin([3, 6, 9, 12]).astype(int)
        df['is_year_end'] = (df['Month'] == 12).astype(int)
        
        # Seasonality features
        df['sin_month'] = np.sin(2 * np.pi * df['Month'] / 12)
        df['cos_month'] = np.cos(2 * np.pi * df['Month'] / 12)
    
    return df

def extract_sector_features(df):
    # Extract sector number from sector column
    df = df.copy()
    if 'sector' in df.columns:
        df['sector'] = df['sector'].str.extract(r'(\d+)').astype(int)
    return df



pros_data = extract_datetime_features(pros_data, date_col='month')
pros_data = extract_sector_features(pros_data)

In [5]:
#Adding exog features based on date which need not be predicted
df = pd.DataFrame(pd.read_csv('./data/sample_submission.csv')['id'])
df["Year"] = df['id'].str[:4].astype(int)
df['Month'] = df['id'].str[5:8]

# Month mapping
month_map = {'Jan':1, 'Feb':2, 'Mar':3, 'Apr':4, 'May':5, 'Jun':6,
            'Jul':7, 'Aug':8, 'Sep':9, 'Oct':10, 'Nov':11, 'Dec':12}
df['Month'] = df['Month'].map(month_map)

# Create datetime index
df['time_index'] = (df['Year'] - 2019) * 12 + df['Month']

# Temporal features
df['quarter'] = (df['Month'] - 1) // 3 + 1
df['is_quarter_end'] = df['Month'].isin([3, 6, 9, 12]).astype(int)
df['is_year_end'] = (df['Month'] == 12).astype(int)

# Seasonality features
df['sin_month'] = np.sin(2 * np.pi * df['Month'] / 12)
df['cos_month'] = np.cos(2 * np.pi * df['Month'] / 12)
df['sector'] = df['id'].str[16:].astype(int)

pros_data = pd.concat([pros_data,df], ignore_index=True)
pros_data.sort_values(['sector','time_index'], inplace=True)

In [6]:
#Sorting the values by sector and time index and then creating a sector hash map to segregate data for each sector . Each sector will have its own model
pros_data.sort_values(['sector', 'time_index'], inplace=True)
sector_map={str(i):pd.DataFrame() for i in range(1, 97)}
for row in pros_data.iterrows():
    sector_map[str(row[1].iloc[1])]= pd.concat([sector_map[str(row[1].iloc[1])],row[1]],ignore_index=True,axis=1)

In [ ]:
import statsmodels.api as sm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

cols = sector_map['1'].columns
cols = [i for i in cols if i not in ['month', 'sector', 'amount_new_house_transactions', 'Year', 'Month', 'time_index', 'id']]

for i in range(1,97):
    #Fixing weird transposed dataframes
    if TARGET_COL not in sector_map[str(i)].columns:
        sector_map[str(i)] = sector_map[str(i)].T

    sector_map[str(i)].fillna(0,inplace=True)

    for tar in cols:
        if tar in ['quarter', 'is_quarter_end', 'is_year_end', 'sin_month', 'cos_month']:
            continue

        model = sm.tsa.SARIMAX(
            endog = sector_map[str(i)][tar][:67],
            order=(1,1,1),
            seasonal_order=(1,1,1,12),
            enforce_stationarity=True, 
            enforce_invertibility=True
        )
        try:
            results = model.fit()
        except:
            print("couldnt do" , tar)
            #If model is too complex try with non seasonal (ARIMA)
            model = sm.tsa.SARIMAX(
                endog = sector_map[str(i)][tar][:67],
                order=(1,1,1),
                seasonal_order=(0,0,0,0),
                enforce_stationarity=True, 
                enforce_invertibility=True
            )
            results = model.fit()

        # print(results.summary())

        forecast = results.get_forecast(steps=12)
        pred = forecast.predicted_mean
        sector_map[str(i)][tar][67:] = pred

1
2
3
4
5
6
7
8
9
10
11
12
couldnt do period_new_house_sell_through_nearby_sectors
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
couldnt do planned_building_area
couldnt do transaction_amount
77
couldnt do transaction_amount
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96


In [ ]:
import statsmodels.api as sm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
preds=[]
ZERO_SECTORS = [12, 19, 26, 33, 39, 41, 44, 49, 52, 53, 58, 67, 72, 73, 74, 75, 82, 87, 89, 95, 96]

for i in range(1,97):
    #Dead sectors which are gonna have 0 sales
    if i in ZERO_SECTORS:
        preds.append(pd.Series([0]*12 , index=range(67,79)))
        continue

    sector_map[str(i)][TARGET_COL] = pd.to_numeric(sector_map[str(i)][TARGET_COL], errors='coerce')

    model = sm.tsa.SARIMAX(
        endog = sector_map[str(i)][TARGET_COL][:67],
        order=(1,1,1),
        seasonal_order=(1,1,1,12),
        exog = sector_map[str(i)][cols][:67]

    )
    results = model.fit()

    # print(results.summary())

    forecast = results.get_forecast(steps=12,exog=sector_map[str(i)][cols][67:])
    pred = forecast.predicted_mean
    #Clipping values which are too low
    for i in range(12):
        if pred[67+i] < 50: pred[67+i] = 0
    preds.append(pred)


In [13]:
#Making the submission file
import csv
month_map = {1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'}
with open('submission.csv', mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['id', 'new_house_transaction_amount'])
    for i in range(12):
        for j in range(1,97):
            year = 2024 + (7+i)//12
            month = month_map[((7+i)%12)+1]
            writer.writerow([f"{year} {month}_sector {j}", int(preds[j-1][67+i])])